# Phase 3: Feature Engineering & MLOps Data Splitting
**AAI-540 ML Ops - Group 8** | **Author:** Jagadeesh Kumar Sellappan

## Overview
This notebook executes the final data engineering transformations. We will denormalize our dataset by joining the core review facts with user and business dimensions to create a single, predictive "wide table." 

Finally, we will split this data into a highly specific MLOps configuration:
* **Train (40%):** For model fitting.
* **Validation (10%):** For hyperparameter tuning.
* **Test (10%):** For final, unbiased evaluation.
* **Production Holdout (40%):** Strictly reserved to simulate real-world inference and test Data Drift monitoring later in the pipeline.

## 1. Environment Setup & Configuration

In [1]:
!pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops sagemaker-studio

Found existing installation: sagemaker-core 2.12.0


Uninstalling sagemaker-core-2.12.0:
  Successfully uninstalled sagemaker-core-2.12.0


Found existing installation: sagemaker-train 1.11.0
Uninstalling sagemaker-train-1.11.0:
  Successfully uninstalled sagemaker-train-1.11.0


Found existing installation: sagemaker-serve 1.11.0
Uninstalling sagemaker-serve-1.11.0:
  Successfully uninstalled sagemaker-serve-1.11.0


Found existing installation: sagemaker-mlops 1.11.0
Uninstalling sagemaker-mlops-1.11.0:
  Successfully uninstalled sagemaker-mlops-1.11.0


Found existing installation: sagemaker_studio 1.1.19
Uninstalling sagemaker_studio-1.1.19:
  Successfully uninstalled sagemaker_studio-1.1.19


In [2]:
!pip install "sagemaker-core==1.0.78" --quiet

In [3]:
!pip install "sagemaker==2.232.2" --no-deps --quiet

In [5]:
!pip install boto3 botocore s3transfer pathos schema importlib-metadata --quiet

In [6]:
!pip install awswrangler --quiet

In [7]:
!pip install -q  seaborn wordcloud 

In [1]:
import awswrangler as wr
import pandas as pd
from sklearn.model_selection import train_test_split
import boto3
import os
import warnings
# warnings.filterwarnings('ignore')

DATABASE = 'yelp_sentiment_db'
BUCKET = 'aai-540-group8-yelp-data-301798465569-us-east-1-an'
SAMPLE_FRAC = 0.02  # Adjust this if needed to hit the ~189k target

print("Environment initialized successfully.")

Environment initialized successfully.


## 2. The Master Join (Feature Engineering)
Instead of joining gigabytes of data in Pandas, we push the compute down to Athena. This query pulls our stratified sample of reviews, engineers the text lengths, and instantly `INNER JOIN`s the business and user metadata.

In [2]:
print("Executing Feature Engineering Master Join in Athena...")

feature_query = f"""
WITH sampled_reviews AS (
    SELECT 
        review_id,
        user_id,
        business_id,
        text,
        LENGTH(text)                    AS text_char_length,
        CARDINALITY(SPLIT(text, ' '))   AS text_word_count,
        useful                          AS review_useful_votes,
        funny                           AS review_funny_votes,
        cool                            AS review_cool_votes,
        CASE
            WHEN stars <= 2 THEN 0
            WHEN stars >= 4 THEN 1
        END AS sentiment_label
    FROM {DATABASE}.reviews_parquet
    WHERE stars != 3
      AND RAND() < {SAMPLE_FRAC}
)
SELECT 
    r.review_id,
    r.text,
    r.text_char_length,
    r.text_word_count,
    r.review_useful_votes,
    r.review_funny_votes,
    r.review_cool_votes,
    b.stars                                         AS business_avg_stars,
    b.review_count                                  AS business_review_count,
    TRIM(SPLIT_PART(b.categories, ',', 1))          AS primary_category,
    b.state                                         AS business_state,
    u.average_stars                                 AS user_avg_stars,
    u.review_count                                  AS user_review_count,
    CASE WHEN u.elite IS NULL 
          OR u.elite = '' THEN 0 ELSE 1 END         AS is_elite,
    r.sentiment_label
FROM sampled_reviews r
INNER JOIN {DATABASE}.businesses_parquet b ON r.business_id = b.business_id
INNER JOIN {DATABASE}.users_parquet u ON r.user_id = u.user_id
"""


# Pull the engineered wide-table into Pandas memory
df_master = wr.athena.read_sql_query(sql=feature_query, database=DATABASE, ctas_approach=False)

# Drop any accidental nulls that slipped through the join to ensure a pristine feature store
df_master = df_master.dropna()

print(f"Feature Engineering Complete. Master Dataset Shape: {df_master.shape}")

Executing Feature Engineering Master Join in Athena...


2026-06-22 02:24:07,991	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 1908387840 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.69gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-22 02:24:08,139	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Feature Engineering Complete. Master Dataset Shape: (126694, 15)


## 3.Amazon SageMaker Feature Store SDK Implementation

In [3]:
import boto3
import sagemaker
from sagemaker.feature_store.feature_group import FeatureGroup
import time
import pandas as pd

print("Initializing SageMaker Feature Store SDK...")

# 1. Setup SageMaker Environment
boto_session = boto3.Session(region_name='us-east-1')
sagemaker_session = sagemaker.Session(boto_session=boto_session)

# Get the execution role
role = sagemaker.get_execution_role()

# Define names
feature_group_name = "yelp-sentiment-features-v1"
record_identifier_name = "review_id"
event_time_feature_name = "EventTime"

# 2. Engineer the Required EventTime Feature
current_time_sec = float(int(time.time()))
df_master['EventTime'] = current_time_sec
df_master['text'] = (df_master['text']
    .astype(str)
    .str.replace('\n', ' ', regex=False)
    .str.replace('\r', '', regex=False)
    .str[:5000]) 
df_master['review_id'] = df_master['review_id'].astype(str)

print(f"Engineered '{event_time_feature_name}' successfully.")

# 3. Initialize and Define the Feature Group
yelp_feature_group = FeatureGroup(
    name=feature_group_name, 
    sagemaker_session=sagemaker_session
)

yelp_feature_group.load_feature_definitions(data_frame=df_master)

# 4. Create the Feature Group in AWS
print(f"Creating Feature Group: {feature_group_name}...")

try:
    yelp_feature_group.create(
        s3_uri=f"s3://{BUCKET}/sagemaker-feature-store-1/", # Offline store location
        record_identifier_name=record_identifier_name,
        event_time_feature_name=event_time_feature_name,
        role_arn=role,
        enable_online_store=False  
    )
    
    # Wait for the Feature Group to be created
    while yelp_feature_group.describe().get("FeatureGroupStatus") == "Creating":
        print("Waiting for Feature Group Creation...")
        time.sleep(5)
        
    print(f"Feature Group Status: {yelp_feature_group.describe().get('FeatureGroupStatus')}")

except Exception as e:
    if "ResourceInUse" in str(e):
        print(f"Feature Group {feature_group_name} already exists. Proceeding to ingestion.")
    else:
        raise e

# 5. Ingest the Data into the Feature Store
print(f"Ingesting entire Master Dataset ({len(df_master):,} rows) into SageMaker Feature Store...")
yelp_feature_group.ingest(
    data_frame=df_master, 
    max_workers=64, 
    wait=True
)
print("Master Data Ingestion Complete!")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Initializing SageMaker Feature Store SDK...


Engineered 'EventTime' successfully.
Creating Feature Group: yelp-sentiment-features-v1...


Waiting for Feature Group Creation...


Waiting for Feature Group Creation...


Waiting for Feature Group Creation...


Waiting for Feature Group Creation...


Feature Group Status: Created
Ingesting entire Master Dataset (126,694 rows) into SageMaker Feature Store...


Master Data Ingestion Complete!


## 4. Read Feature and Split train/test/val/Prod

In [5]:
from sklearn.model_selection import train_test_split
import time

print("Extracting feature data from SageMaker Feature Store via Athena...")

# 1. Initialize the Athena query builder from your Feature Group
query = yelp_feature_group.athena_query()
table_name = query.table_name

# 2. Write the SQL query to pull all records
query_string = query_string = f"""
    SELECT *
    FROM "{table_name}"
    WHERE is_deleted = false
"""

# 3. Execute the query
# Athena requires a temporary S3 folder to drop query results into
output_location = f's3://{BUCKET}/athena-query-results/'
query.run(query_string=query_string, output_location=output_location)

print("Waiting for Athena query to complete (this may take 1-2 minutes)...")
query.wait()

# 4. Load the results directly into a Pandas DataFrame
df_extracted_features = query.as_dataframe()
print(f"Successfully extracted {len(df_extracted_features):,} rows from the Feature Store.")


# Perform the MLOps Splits

print("\nInitiating Stratified MLOps Data Splits from feature data...")

# 1. Isolate the 40% Production Holdout first
df_modeling, df_production = train_test_split(
    df_extracted_features, 
    test_size=0.40, 
    stratify=df_extracted_features['sentiment_label'], 
    random_state=42
)

# 2. Split the modeling data into Train (40% of total) and Temp (20% of total)
df_train, df_temp = train_test_split(
    df_modeling, 
    train_size=(40/60), 
    stratify=df_modeling['sentiment_label'], 
    random_state=42
)

# 3. Split the Temp data perfectly in half to get Validation (10%) and Test (10%)
df_val, df_test = train_test_split(
    df_temp, 
    test_size=0.50, 
    stratify=df_temp['sentiment_label'], 
    random_state=42
)

print("\n Final Split Distribution:")
print(f"  -> Training (40%):    {len(df_train):,} rows")
print(f"  -> Validation (10%):  {len(df_val):,} rows")
print(f"  -> Testing (10%):     {len(df_test):,} rows")
print(f"  -> Production (40%):  {len(df_production):,} rows")

Extracting feature data from SageMaker Feature Store via Athena...


Waiting for Athena query to complete (this may take 1-2 minutes)...


Successfully extracted 126,694 rows from the Feature Store.

Initiating Stratified MLOps Data Splits from feature data...

 Final Split Distribution:
  -> Training (40%):    50,677 rows
  -> Validation (10%):  12,669 rows
  -> Testing (10%):     12,670 rows
  -> Production (40%):  50,678 rows


In [8]:
# save all 4 to S3 as Parquet
print("\nSaving splits to S3...")
splits = {
    'train':      df_train,
    'validation': df_val,
    'test':       df_test,
    'production': df_production
}

for split_name, split_df in splits.items():
    s3_path = f"s3://{BUCKET}/data/splits/{split_name}/{split_name}.parquet"
    wr.s3.to_parquet(df=split_df, path=s3_path, index=False)
    print(f"  {split_name:12s} → {s3_path}  ({len(split_df):,} rows)")

print("\n All splits saved to S3 successfully.")


Saving splits to S3...


  train        → s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/data/splits/train/train.parquet  (50,677 rows)


  validation   → s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/data/splits/validation/validation.parquet  (12,669 rows)


  test         → s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/data/splits/test/test.parquet  (12,670 rows)


  production   → s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/data/splits/production/production.parquet  (50,678 rows)

 All splits saved to S3 successfully.


In [9]:
print("\nClass balance verification across all splits:")
for name, split_df in splits.items():
    neg = (split_df['sentiment_label']==0).mean()
    print(f"  {name:12s}: {len(split_df):>7,} rows | "
          f"Neg: {neg:.1%} | Pos: {1-neg:.1%}")


Class balance verification across all splits:
  train       :  50,677 rows | Neg: 25.7% | Pos: 74.3%
  validation  :  12,669 rows | Neg: 25.7% | Pos: 74.3%
  test        :  12,670 rows | Neg: 25.7% | Pos: 74.3%
  production  :  50,678 rows | Neg: 25.7% | Pos: 74.3%


## Clean up

In [ ]:
# # Run this cell FIRST before rerunning the notebook
# import boto3
# import sagemaker

# # Region standardized to us-east-1 to match the unified project bucket
# boto_session = boto3.Session(region_name='us-east-1')
# sagemaker_session = sagemaker.Session(boto_session=boto_session)

# from sagemaker.feature_store.feature_group import FeatureGroup

# feature_group_name = "yelp-sentiment-features-v1"

# yelp_feature_group = FeatureGroup(
#     name=feature_group_name,
#     sagemaker_session=sagemaker_session
# )

# try:
#     yelp_feature_group.delete()
#     print(f"✅ Feature Group '{feature_group_name}' deleted successfully.")
#     print("   Wait ~30 seconds before rerunning the notebook.")
# except Exception as e:
#     print(f"⚠️ {e}")

In [ ]:
# import awswrangler as wr

# # Unified project bucket (us-east-1) - standardized across all notebooks
# BUCKET = 'aai-540-group8-yelp-data-301798465569-us-east-1-an'

# # Delete Feature Store offline data
# wr.s3.delete_objects(
#     path=f"s3://{BUCKET}/sagemaker-feature-store/",
# )
# print("✅ S3 Feature Store data deleted.")

# # Delete old splits too since we're regenerating
# wr.s3.delete_objects(
#     path=f"s3://{BUCKET}/data/splits/"
# )
# print("✅ Old splits deleted.")

*** SIGTERM received at time=1782095003 on cpu 1 ***
PC: @     0x7fd636c93072  (unknown)  epoll_wait
[2026-06-22 02:23:23,196 E 763 763] logging.cc:474: *** SIGTERM received at time=1782095003 on cpu 1 ***
[2026-06-22 02:23:23,196 E 763 763] logging.cc:474: PC: @     0x7fd636c93072  (unknown)  epoll_wait
